# 🍆 Bilau Animator — Game-Quality Sprite FX

Transforma PNGs 256×256 em GIFs animados com efeitos de jogo reais:

- 🫀 **Idle breathing** — escala + squash/stretch sutil
- ✨ **Particle aura** — partículas flutuando ao redor
- 🔥 **Energy aura** — brilho pulsante colorido ao redor do sprite
- 💫 **Floating** — flutua suavemente pra cima e baixo com sombra
- ⚡ **Power surge** — tremor + flash + partículas intensas
- 👑 **Divine** — floating + aura dourada + partículas + halo

**Não precisa de GPU!** Roda em CPU, é instantâneo.

In [ ]:
#@title 1️⃣ Setup
!pip install -q Pillow numpy
import os
os.makedirs("/content/input", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)
print("✅ Pronto! Faça upload das PNGs ou GIFs em /content/input/")

In [ ]:
#@title 2️⃣ Upload das PNGs / GIFs
from google.colab import files
from PIL import Image
from IPython.display import display

uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"/content/input/{name}", "wb") as f:
        f.write(data)
    print(f"  ✅ {name}")

sprites = sorted([f for f in os.listdir("/content/input") if f.lower().endswith((".png", ".gif"))])
print(f"\n📦 {len(sprites)} sprites")
if sprites:
    display(Image.open(f"/content/input/{sprites[0]}"))

In [ ]:
#@title 3️⃣ Motor de animação (rodar uma vez)
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
import numpy as np
import math, random, struct, zlib, io, base64

SIZE = 256

def load_sprite(path):
    """Carrega PNG (1 frame) ou GIF (N frames). Retorna lista de RGBA frames."""
    img = Image.open(path)
    frames = []
    try:
        while True:
            frame = img.copy().convert("RGBA")
            if frame.size != (SIZE, SIZE):
                frame = frame.resize((SIZE, SIZE), Image.NEAREST)
            frames.append(frame)
            img.seek(img.tell() + 1)
    except EOFError:
        pass
    if not frames:
        # fallback
        frame = img.convert("RGBA")
        if frame.size != (SIZE, SIZE):
            frame = frame.resize((SIZE, SIZE), Image.NEAREST)
        frames = [frame]
    return frames


# ─── Helpers pra aplicar efeitos por frame ───
def _apply_to_frames(src_frames, effect_fn, num_effect_frames):
    """Aplica effect_fn a cada source frame × cada effect step.
    effect_fn(sprite, i, num_effect_frames) → single RGBA frame
    Se input é GIF (N source frames), saída tem N × num_effect_frames frames,
    ciclando pelos source frames pra cada step do efeito.
    Se input é PNG (1 frame), saída tem num_effect_frames frames normal."""
    out = []
    for i in range(num_effect_frames):
        src = src_frames[i % len(src_frames)]
        out.append(effect_fn(src, i, num_effect_frames))
    return out


# ─── AURA GLOW ───
def make_aura(sprite, color=(180, 120, 255), radius=6, intensity=0.6):
    """Cria uma aura brilhante ao redor do sprite."""
    alpha = sprite.split()[3]
    expanded = alpha.filter(ImageFilter.MaxFilter(radius * 2 + 1))
    glow = expanded.filter(ImageFilter.GaussianBlur(radius))
    aura = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
    glow_arr = np.array(glow).astype(float) / 255.0 * intensity
    aura_arr = np.zeros((SIZE, SIZE, 4), dtype=np.uint8)
    aura_arr[:, :, 0] = (glow_arr * color[0]).astype(np.uint8)
    aura_arr[:, :, 1] = (glow_arr * color[1]).astype(np.uint8)
    aura_arr[:, :, 2] = (glow_arr * color[2]).astype(np.uint8)
    aura_arr[:, :, 3] = (glow_arr * 255).astype(np.uint8)
    return Image.fromarray(aura_arr)


# ─── PARTÍCULAS ───
class Particle:
    def __init__(self, color, size_range=(2, 5), area="around"):
        self.color = color
        self.size = random.randint(*size_range)
        if area == "around":
            angle = random.uniform(0, 2 * math.pi)
            dist = random.uniform(60, 120)
            self.x = SIZE // 2 + dist * math.cos(angle)
            self.y = SIZE // 2 + dist * math.sin(angle)
        elif area == "bottom":
            self.x = random.uniform(40, SIZE - 40)
            self.y = random.uniform(SIZE * 0.7, SIZE)
        elif area == "full":
            self.x = random.uniform(20, SIZE - 20)
            self.y = random.uniform(20, SIZE - 20)
        self.vx = random.uniform(-0.5, 0.5)
        self.vy = random.uniform(-2.0, -0.5)
        self.life = random.uniform(0.3, 1.0)
        self.phase = random.uniform(0, 2 * math.pi)

    def update(self, t):
        self.x += self.vx
        self.y += self.vy
        self.life -= 0.04

    def draw(self, draw, t):
        if self.life <= 0:
            return
        alpha = int(255 * self.life * (0.5 + 0.5 * math.sin(t * 4 + self.phase)))
        alpha = max(0, min(255, alpha))
        c = (*self.color, alpha)
        s = self.size
        draw.ellipse([self.x - s, self.y - s, self.x + s, self.y + s], fill=c)


def generate_particles(n, color, area="around", size_range=(2, 5)):
    return [Particle(color, size_range, area) for _ in range(n)]


# ─── SOMBRA ───
def make_shadow(sprite, y_offset=0):
    """Cria sombra embaixo do sprite."""
    alpha = sprite.split()[3]
    shadow = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
    shadow_alpha = alpha.resize((int(SIZE * 0.7), int(SIZE * 0.15)), Image.LANCZOS)
    shadow_alpha = shadow_alpha.filter(ImageFilter.GaussianBlur(4))
    shadow_img = Image.new("RGBA", shadow_alpha.size, (0, 0, 0, 80))
    shadow_img.putalpha(shadow_alpha)
    sx = (SIZE - shadow_img.width) // 2
    sy = SIZE - 30 + y_offset
    shadow.paste(shadow_img, (sx, sy), shadow_img)
    return shadow


# ─── HALO ───
def make_halo(t, color=(255, 215, 0), radius=90):
    """Desenha um halo/anel brilhante acima do sprite."""
    halo = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
    draw = ImageDraw.Draw(halo)
    pulse = 0.7 + 0.3 * math.sin(t * 3)
    alpha = int(180 * pulse)
    cx, cy = SIZE // 2, 30
    r = int(radius * 0.4)
    for w in range(3, 0, -1):
        a = int(alpha * (w / 3))
        c = (*color, a)
        draw.ellipse([cx - r - w, cy - r//3 - w, cx + r + w, cy + r//3 + w], outline=c, width=2)
    return halo


# ════════════════════════════════════════════
# ANIMATION PRESETS
# Todas aceitam src_frames (lista de RGBA) e retornam lista de frames animados.
# Se input é GIF, os source frames são ciclados dentro de cada animação.
# ════════════════════════════════════════════

def anim_idle(src_frames, num_frames=12):
    """🫀 Idle breathing — squash/stretch sutil."""
    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        sx = 1.0 - 0.02 * math.sin(t)
        sy = 1.0 + 0.02 * math.sin(t)
        nw, nh = int(SIZE * sx), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)
        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        ox = (SIZE - nw) // 2
        oy = SIZE - nh
        frame.paste(scaled, (ox, oy), scaled)
        return frame
    return _apply_to_frames(src_frames, effect, num_frames)


def anim_particle_aura(src_frames, num_frames=16, color=(180, 120, 255), n_particles=25, area="around"):
    """✨ Partículas flutuando ao redor do sprite."""
    random.seed(42)
    all_particles = []
    for wave in range(3):
        all_particles.append(generate_particles(n_particles, color, area))

    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        sy = 1.0 + 0.015 * math.sin(t)
        sx = 1.0 - 0.015 * math.sin(t)
        nw, nh = int(SIZE * sx), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)

        particle_layer = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        pdraw = ImageDraw.Draw(particle_layer)
        wave_idx = i % 3
        for p in all_particles[wave_idx]:
            p.update(t)
            p.draw(pdraw, t + i * 0.5)

        frame = Image.alpha_composite(frame, particle_layer)
        ox = (SIZE - nw) // 2
        oy = SIZE - nh
        frame.paste(scaled, (ox, oy), scaled)

        front_layer = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        fdraw = ImageDraw.Draw(front_layer)
        next_wave = (wave_idx + 1) % 3
        for p in all_particles[next_wave]:
            p.draw(fdraw, t + i * 0.3)
        frame = Image.alpha_composite(frame, front_layer)
        return frame

    return _apply_to_frames(src_frames, effect, num_frames)


def anim_energy_aura(src_frames, num_frames=14, color=(100, 180, 255)):
    """🔥 Aura de energia pulsante ao redor."""
    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

        pulse = 0.4 + 0.6 * (0.5 + 0.5 * math.sin(t))
        radius = int(4 + 4 * math.sin(t))
        aura = make_aura(sprite, color, radius=max(3, radius), intensity=pulse * 0.8)
        frame = Image.alpha_composite(frame, aura)

        sy = 1.0 + 0.02 * math.sin(t)
        sx = 1.0 - 0.02 * math.sin(t)
        nw, nh = int(SIZE * sx), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)
        ox = (SIZE - nw) // 2
        oy = SIZE - nh
        frame.paste(scaled, (ox, oy), scaled)

        brightness = 1.0 + 0.15 * math.sin(t)
        bright_sprite = ImageEnhance.Brightness(frame).enhance(brightness)
        bright_sprite.putalpha(frame.split()[3])
        return bright_sprite

    return _apply_to_frames(src_frames, effect, num_frames)


def anim_floating(src_frames, num_frames=14, float_height=10):
    """💫 Flutuando com sombra embaixo."""
    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        dy = int(float_height * math.sin(t))

        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

        shadow_scale = 0.7 + 0.3 * (1 - (math.sin(t) + 1) / 2)
        shadow = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        sw, sh = int(80 * shadow_scale), int(15 * shadow_scale)
        shadow_draw = ImageDraw.Draw(shadow)
        shadow_alpha = int(60 * shadow_scale)
        sx = SIZE // 2
        sy_pos = SIZE - 20
        shadow_draw.ellipse([sx - sw, sy_pos - sh, sx + sw, sy_pos + sh], fill=(0, 0, 0, shadow_alpha))
        shadow = shadow.filter(ImageFilter.GaussianBlur(3))
        frame = Image.alpha_composite(frame, shadow)

        sy = 1.0 + 0.01 * math.sin(t)
        sx_s = 1.0 - 0.01 * math.sin(t)
        nw, nh = int(SIZE * sx_s), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)
        ox = (SIZE - nw) // 2
        oy = SIZE - nh - 15 - abs(dy)
        frame.paste(scaled, (ox, oy), scaled)
        return frame

    return _apply_to_frames(src_frames, effect, num_frames)


def anim_power_surge(src_frames, num_frames=12, color=(255, 80, 80)):
    """⚡ Tremor + flash + partículas intensas."""
    random.seed(42)
    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

        pulse = 0.5 + 0.5 * math.sin(t * 2)
        aura = make_aura(sprite, color, radius=int(5 + 5 * pulse), intensity=pulse * 0.9)
        frame = Image.alpha_composite(frame, aura)

        particle_layer = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        pdraw = ImageDraw.Draw(particle_layer)
        n_p = 15 + int(10 * pulse)
        for _ in range(n_p):
            px = SIZE // 2 + random.randint(-70, 70)
            py = random.randint(30, SIZE - 30)
            py -= int(40 * (i / n))
            ps = random.randint(1, 4)
            pa = random.randint(80, 220)
            pdraw.ellipse([px - ps, py - ps, px + ps, py + ps], fill=(*color, pa))
        frame = Image.alpha_composite(frame, particle_layer)

        shake_x = random.randint(-3, 3)
        shake_y = random.randint(-2, 2)
        sy = 1.0 + 0.03 * math.sin(t * 2)
        sx_s = 1.0 - 0.03 * math.sin(t * 2)
        nw, nh = int(SIZE * sx_s), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)
        ox = (SIZE - nw) // 2 + shake_x
        oy = SIZE - nh + shake_y
        frame.paste(scaled, (ox, oy), scaled)

        brightness = 1.0 + 0.25 * pulse
        bright = ImageEnhance.Brightness(frame).enhance(brightness)
        bright.putalpha(frame.split()[3])
        return bright

    return _apply_to_frames(src_frames, effect, num_frames)


def anim_divine(src_frames, num_frames=16, aura_color=(255, 215, 0), particle_color=(255, 255, 180)):
    """👑 Floating + aura dourada + partículas + halo — pras evoluções finais."""
    random.seed(42)
    def effect(sprite, i, n):
        t = 2 * math.pi * i / n
        frame = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

        dy = int(8 * math.sin(t))

        # Sombra
        shadow_scale = 0.7 + 0.3 * (1 - (math.sin(t) + 1) / 2)
        shadow_layer = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        sdraw = ImageDraw.Draw(shadow_layer)
        sw = int(70 * shadow_scale)
        sh_s = int(12 * shadow_scale)
        sa = int(50 * shadow_scale)
        sdraw.ellipse([SIZE//2 - sw, SIZE - 25 - sh_s, SIZE//2 + sw, SIZE - 25 + sh_s], fill=(0, 0, 0, sa))
        shadow_layer = shadow_layer.filter(ImageFilter.GaussianBlur(3))
        frame = Image.alpha_composite(frame, shadow_layer)

        # Aura
        pulse = 0.5 + 0.5 * math.sin(t)
        aura = make_aura(sprite, aura_color, radius=int(5 + 4 * pulse), intensity=pulse * 0.7)
        aura_shifted = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        aura_shifted.paste(aura, (0, -15 - abs(dy)), aura)
        frame = Image.alpha_composite(frame, aura_shifted)

        # Partículas
        particle_layer = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        pdraw = ImageDraw.Draw(particle_layer)
        for _ in range(20):
            px = SIZE // 2 + random.randint(-80, 80)
            py = random.randint(20, SIZE - 40)
            py -= int(30 * ((i + random.random() * 4) / n))
            ps = random.randint(1, 3)
            pa = random.randint(60, 200)
            twinkle = 0.5 + 0.5 * math.sin(t * 3 + px * 0.1)
            pa = int(pa * twinkle)
            pdraw.ellipse([px - ps, py - ps, px + ps, py + ps], fill=(*particle_color, pa))
        frame = Image.alpha_composite(frame, particle_layer)

        # Sprite flutuando
        sy = 1.0 + 0.015 * math.sin(t)
        sx_s = 1.0 - 0.015 * math.sin(t)
        nw, nh = int(SIZE * sx_s), int(SIZE * sy)
        scaled = sprite.resize((nw, nh), Image.NEAREST)
        ox = (SIZE - nw) // 2
        oy = SIZE - nh - 15 - abs(dy)
        frame.paste(scaled, (ox, oy), scaled)

        # Halo
        halo = make_halo(t, aura_color)
        halo_shifted = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))
        halo_shifted.paste(halo, (0, -15 - abs(dy)), halo)
        frame = Image.alpha_composite(frame, halo_shifted)

        # Brilho
        brightness = 1.0 + 0.1 * math.sin(t)
        bright = ImageEnhance.Brightness(frame).enhance(brightness)
        bright.putalpha(frame.split()[3])
        return bright

    return _apply_to_frames(src_frames, effect, num_frames)


# ─── REGISTRY ───
ANIMATIONS = {
    "idle":           lambda s, n=12: anim_idle(s, n),
    "particle_aura":  lambda s, n=16: anim_particle_aura(s, n, color=(180, 120, 255)),
    "energy_aura":    lambda s, n=14: anim_energy_aura(s, n, color=(100, 180, 255)),
    "energy_red":     lambda s, n=14: anim_energy_aura(s, n, color=(255, 80, 60)),
    "energy_gold":    lambda s, n=14: anim_energy_aura(s, n, color=(255, 200, 50)),
    "energy_green":   lambda s, n=14: anim_energy_aura(s, n, color=(80, 255, 120)),
    "floating":       lambda s, n=14: anim_floating(s, n),
    "power_surge":    lambda s, n=12: anim_power_surge(s, n, color=(255, 80, 80)),
    "power_blue":     lambda s, n=12: anim_power_surge(s, n, color=(80, 150, 255)),
    "divine":         lambda s, n=16: anim_divine(s, n),
    "divine_red":     lambda s, n=16: anim_divine(s, n, aura_color=(255, 50, 50), particle_color=(255, 150, 150)),
    "divine_blue":    lambda s, n=16: anim_divine(s, n, aura_color=(80, 150, 255), particle_color=(180, 210, 255)),
}


# ─── SAVE GIF ───
def save_gif(frames, path, duration=100):
    gif_frames = []
    for f in frames:
        canvas = Image.new("RGBA", f.size, (0, 0, 0, 0))
        canvas = Image.alpha_composite(canvas, f)
        alpha = canvas.split()[3]
        p = canvas.convert("RGB").quantize(colors=255, method=2)
        mask = Image.eval(alpha, lambda a: 255 if a <= 128 else 0)
        p.paste(255, mask)
        gif_frames.append(p)
    gif_frames[0].save(
        path, save_all=True, append_images=gif_frames[1:],
        duration=duration, loop=0, transparency=255, disposal=2,
    )


def preview_gif(path):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    from IPython.display import display, HTML
    display(HTML(f'<img src="data:image/gif;base64,{b64}" style="width:256px;height:256px;image-rendering:pixelated;background:#1a1a2e;">'))


print("✅ Motor de animação carregado! (suporta PNG e GIF como input)")
print("\nAnimações disponíveis:")
for name in ANIMATIONS:
    print(f"  • {name}")
print("\n💡 GIFs de input terão seus frames preservados + efeitos aplicados por cima")

In [ ]:
#@title 4️⃣ Testar UM sprite — preview de todas as animações
from IPython.display import display, HTML

ARQUIVO = ""  #@param {type:"string"}
VELOCIDADE_MS = 100  #@param {type:"slider", min:50, max:200, step:10}

if not ARQUIVO:
    sprites = sorted([f for f in os.listdir("/content/input") if f.lower().endswith((".png", ".gif"))])
    ARQUIVO = sprites[0] if sprites else None
if not ARQUIVO:
    raise FileNotFoundError("Nenhum PNG/GIF em /content/input/")

src_frames = load_sprite(f"/content/input/{ARQUIVO}")
print(f"🎬 Testando todas as animações com: {ARQUIVO} ({len(src_frames)} frame(s) de input)\n")

html = '<div style="display:flex;flex-wrap:wrap;gap:16px;background:#111;padding:16px;border-radius:12px;">'
for name, func in ANIMATIONS.items():
    frames = func(src_frames)
    tmp_path = f"/content/output/_preview_{name}.gif"
    save_gif(frames, tmp_path, duration=VELOCIDADE_MS)
    with open(tmp_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html += f'''
    <div style="text-align:center;">
        <img src="data:image/gif;base64,{b64}" style="width:160px;height:160px;image-rendering:pixelated;">
        <br><small style="color:#aaa;font-family:monospace;">{name}</small>
    </div>'''
html += '</div>'
display(HTML(html))
print("\n☝️ Escolha a animação que mais curtir pro próximo passo!")

In [ ]:
#@title 5️⃣ Animar UM sprite específico
ARQUIVO = ""           #@param {type:"string"}
ANIMACAO = "idle"      #@param ["idle", "particle_aura", "energy_aura", "energy_red", "energy_gold", "energy_green", "floating", "power_surge", "power_blue", "divine", "divine_red", "divine_blue"]
VELOCIDADE_MS = 100    #@param {type:"slider", min:50, max:200, step:10}

if not ARQUIVO:
    sprites = sorted([f for f in os.listdir("/content/input") if f.lower().endswith((".png", ".gif"))])
    ARQUIVO = sprites[0] if sprites else None

src_frames = load_sprite(f"/content/input/{ARQUIVO}")
frames = ANIMATIONS[ANIMACAO](src_frames)
# Output é sempre .gif
out_name = os.path.splitext(ARQUIVO)[0] + ".gif"
out_path = f"/content/output/{out_name}"
save_gif(frames, out_path, duration=VELOCIDADE_MS)
print(f"✅ {out_name} — {ANIMACAO} ({len(src_frames)} frame(s) de input → {len(frames)} frames de output)")
preview_gif(out_path)

In [ ]:
#@title 6️⃣ Batch — animar TODOS com animação por fase
import re

VELOCIDADE_MS = 100  #@param {type:"slider", min:50, max:200, step:10}

# Mapeamento automático: número no arquivo → animação
# Customize como quiser!
def get_anim_for_file(filename):
    m = re.search(r'(\d+)', filename)
    if m:
        n = int(m.group(1))
        if n <= 3:  return "idle"            # Pequênis, Bilinho, Misterioso
        if n <= 5:  return "floating"         # Definhado, Bilargo
        if n <= 8:  return "particle_aura"    # Morto, Fantasma, Bilão
        if n <= 10: return "energy_aura"      # Bilailish, Super-Bilau
        if n <= 13: return "energy_green"     # PM, Patriota, Guerra
        if n <= 16: return "energy_gold"      # Bora, Bilacula, Akatsuki
        if n <= 20: return "energy_red"       # SSJ, SSJ3
        if n <= 25: return "power_surge"      # SSJ4, MUI, Mega
        if n <= 30: return "power_blue"       # Proto, Bilonic, etc
        if n <= 35: return "divine"           # Bilope, Bilgneto...
        return "divine_red"                   # Einstein, Vinci, Bileus
    return "idle"

sprites = sorted([f for f in os.listdir("/content/input") if f.lower().endswith((".png", ".gif"))])
print(f"🎬 Animando {len(sprites)} sprites...\n")

for i, name in enumerate(sprites):
    anim_type = get_anim_for_file(name)
    src_frames = load_sprite(f"/content/input/{name}")
    frames = ANIMATIONS[anim_type](src_frames)
    out_name = os.path.splitext(name)[0] + ".gif"
    save_gif(frames, f"/content/output/{out_name}", duration=VELOCIDADE_MS)
    src_info = f", {len(src_frames)}f input" if len(src_frames) > 1 else ""
    print(f"  ✅ [{i+1}/{len(sprites)}] {name} → {out_name} ({anim_type}{src_info})")

print(f"\n🎉 Pronto! {len(sprites)} GIFs em /content/output/")

In [ ]:
#@title 7️⃣ Preview de todos os resultados
from IPython.display import display, HTML

gifs = sorted([f for f in os.listdir("/content/output") if f.endswith(".gif") and not f.startswith("_")])
print(f"👁️ {len(gifs)} GIFs:\n")

html = '<div style="display:flex;flex-wrap:wrap;gap:12px;background:#111;padding:16px;border-radius:12px;">'
for name in gifs:
    with open(f"/content/output/{name}", "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html += f'''
    <div style="text-align:center;">
        <img src="data:image/gif;base64,{b64}" style="width:128px;height:128px;image-rendering:pixelated;">
        <br><small style="color:#aaa;">{name}</small>
    </div>'''
html += '</div>'
display(HTML(html))

In [ ]:
#@title 8️⃣ Baixar ZIP
import shutil
shutil.make_archive("/content/bilau_animated", "zip", "/content/output")
print("✅ bilau_animated.zip criado!")
try:
    from google.colab import files
    files.download("/content/bilau_animated.zip")
except:
    print("📁 Baixe pelo painel de arquivos → bilau_animated.zip")

## 📋 Referência

### Input suportado
| Formato | Comportamento |
|---------|--------------|
| **PNG** | Imagem estática — efeitos VFX são aplicados normalmente |
| **GIF** | Animação existente — cada frame do GIF recebe os efeitos VFX por cima, preservando a animação original |

### Animações
| Animação | Efeito | Ideal pra |
|----------|--------|----------|
| `idle` | Respiração squash/stretch | Evoluções simples (1-3) |
| `floating` | Flutua com sombra | Evoluções fantasma/leves (4-5) |
| `particle_aura` | Partículas flutuando | Evoluções místicas (6-8) |
| `energy_aura` | Aura azul pulsante | Evoluções fortes (9-10) |
| `energy_red` | Aura vermelha | SSJ / poder (17-20) |
| `energy_gold` | Aura dourada | Evoluções nobres (14-16) |
| `energy_green` | Aura verde | Evoluções naturais (11-13) |
| `power_surge` | Tremor + flash + partículas | Evoluções intensas (21-30) |
| `power_blue` | Surge azul | Evoluções tech/robô |
| `divine` | Float + halo + aura dourada | Evoluções divinas (31-35) |
| `divine_red` | Divino vermelho | Evoluções finais (36-40) |
| `divine_blue` | Divino azul | Variante |

### Dicas
- Cell 4 mostra TODAS as animações lado a lado pra comparar
- Cell 5 testa uma específica antes do batch
- Cell 6 faz batch com animação auto-atribuída por fase
- Customize o `get_anim_for_file()` na cell 6 pro mapping que quiser
- **GIF inputs**: os source frames são ciclados dentro da animação — ex: um GIF de 4 frames com `idle` (12 effect frames) vai ciclar os 4 frames 3 vezes, aplicando os efeitos VFX em cada um

# 🍆 Bilau Animator — AI-Powered Sprite Animation

Usa **Stable Video Diffusion (img2vid)** pra dar vida aos sprites do Bilau Clicker.
A IA gera movimento real: auras, expressões, maneirismos, efeitos visuais.

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Rode as células em ordem
3. Upload das PNGs 256×256
4. A IA anima cada sprite com movimento real
5. Baixe os GIFs

In [ ]:
#@title 1️⃣ Verificar GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 2️⃣ Instalar dependências
!pip install -q diffusers transformers accelerate safetensors pillow
!pip install -q peft
import os
os.makedirs("/content/input", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)
print("✅ Dependências instaladas!")

In [ ]:
#@title 3️⃣ Carregar modelo SVD (Stable Video Diffusion — img2vid)
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe = pipe.to("cuda")
pipe.enable_model_cpu_offload()     # economiza VRAM
pipe.unet.enable_forward_chunking() # mais economia

print("✅ Stable Video Diffusion carregado!")
print("   Esse modelo pega uma imagem estática e gera um vídeo curto com movimento real.")

In [ ]:
#@title 4️⃣ Upload das PNGs
from google.colab import files
from PIL import Image
from IPython.display import display

uploaded = files.upload()
for name, data in uploaded.items():
    dest = f"/content/input/{name}"
    with open(dest, "wb") as f:
        f.write(data)
    print(f"  ✅ {name}")

pngs = sorted([f for f in os.listdir("/content/input") if f.lower().endswith(".png")])
print(f"\n📦 {len(pngs)} sprites prontos")
if pngs:
    print("\n👁️ Preview do primeiro:")
    display(Image.open(f"/content/input/{pngs[0]}").resize((256,256), Image.NEAREST))

In [ ]:
#@title 5️⃣ Helpers: preparar imagem + converter pra GIF
from PIL import Image
import numpy as np

def prepare_image(path, size=512):
    """Carrega PNG e prepara pro SVD (precisa ser 512x512 ou 1024x576)."""
    img = Image.open(path).convert("RGB")
    # SVD espera 1024x576 por padrão, mas 512x512 funciona
    img = img.resize((size, size), Image.LANCZOS)
    return img


def frames_to_gif(frames, output_path, target_size=256, duration=120):
    """Converte lista de frames PIL pra GIF animado."""
    gif_frames = []
    for frame in frames:
        # Redimensionar de volta pro tamanho do sprite com NEAREST (preserva pixel art)
        f = frame.resize((target_size, target_size), Image.NEAREST)
        gif_frames.append(f.convert("P", palette=Image.ADAPTIVE, colors=256))

    gif_frames[0].save(
        output_path,
        save_all=True,
        append_images=gif_frames[1:],
        duration=duration,
        loop=0,
        disposal=2,
    )


def preview_gif(path):
    """Mostra GIF inline no notebook."""
    import base64
    from IPython.display import display, HTML
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{b64}" style="width:256px;height:256px;image-rendering:pixelated;">'))


print("✅ Helpers carregados!")

In [ ]:
#@title 6️⃣ Animar UM sprite (teste)
from IPython.display import display

ARQUIVO = ""           #@param {type:"string"}
SEED = 42              #@param {type:"integer"}
NUM_FRAMES = 14        #@param {type:"slider", min:6, max:25, step:1}
MOTION_BUCKET = 80     #@param {type:"slider", min:30, max:200, step:10}
FPS = 8                #@param {type:"slider", min:4, max:16, step:1}
DECODE_CHUNK = 4       #@param {type:"slider", min:2, max:8, step:1}

# motion_bucket_id controla a intensidade do movimento:
#   30-60  = movimento sutil (respiração, brilho)
#   80-120 = movimento médio (auras, balanço)
#   150+   = movimento intenso (ação, tremor)

if not ARQUIVO:
    pngs = sorted([f for f in os.listdir("/content/input") if f.lower().endswith(".png")])
    ARQUIVO = pngs[0] if pngs else None

if not ARQUIVO:
    raise FileNotFoundError("Nenhum PNG em /content/input/")

print(f"🎬 Animando: {ARQUIVO}")
print(f"   Frames: {NUM_FRAMES}, Motion: {MOTION_BUCKET}, FPS: {FPS}")

image = prepare_image(f"/content/input/{ARQUIVO}", size=512)
generator = torch.Generator("cuda").manual_seed(SEED)

with torch.inference_mode():
    result = pipe(
        image,
        num_frames=NUM_FRAMES,
        decode_chunk_size=DECODE_CHUNK,
        motion_bucket_id=MOTION_BUCKET,
        noise_aug_strength=0.02,   # baixo pra manter fidelidade ao sprite original
        generator=generator,
    )

frames = result.frames[0]  # lista de PIL Images
print(f"   ✅ {len(frames)} frames gerados")

# Fazer loop: adiciona frames em reverso pra criar animação cíclica suave
looped = list(frames) + list(reversed(frames[1:-1]))

out_name = ARQUIVO.replace(".png", ".gif")
out_path = f"/content/output/{out_name}"
frame_duration = 1000 // FPS
frames_to_gif(looped, out_path, target_size=256, duration=frame_duration)

print(f"   ✅ Salvo: {out_name} ({len(looped)} frames, {frame_duration}ms each)")
print("\n👁️ Preview:")
preview_gif(out_path)

In [ ]:
#@title 7️⃣ Animar TODOS os sprites (batch)
import re

SEED = 42              #@param {type:"integer"}
NUM_FRAMES = 14        #@param {type:"slider", min:6, max:25, step:1}
FPS = 8                #@param {type:"slider", min:4, max:16, step:1}
DECODE_CHUNK = 4       #@param {type:"slider", min:2, max:8, step:1}

# Motion intensity por fase (baseado no número no nome do arquivo)
# Quanto maior o número, mais movimento
def get_motion_for_file(filename):
    m = re.search(r'(\d+)', filename)
    if m:
        n = int(m.group(1))
        if n <= 5:  return 40   # sutil — respiração leve
        if n <= 10: return 60   # leve — balanço suave
        if n <= 15: return 80   # médio — aura suave
        if n <= 20: return 100  # forte — aura visível
        if n <= 30: return 130  # intenso — efeitos
        return 170              # máximo — divino/épico
    return 80

pngs = sorted([f for f in os.listdir("/content/input") if f.lower().endswith(".png")])
print(f"🎬 Animando {len(pngs)} sprites com IA...\n")

for i, name in enumerate(pngs):
    motion = get_motion_for_file(name)
    print(f"  🔄 [{i+1}/{len(pngs)}] {name} (motion={motion})...", end=" ")

    image = prepare_image(f"/content/input/{name}", size=512)
    generator = torch.Generator("cuda").manual_seed(SEED + i)

    with torch.inference_mode():
        result = pipe(
            image,
            num_frames=NUM_FRAMES,
            decode_chunk_size=DECODE_CHUNK,
            motion_bucket_id=motion,
            noise_aug_strength=0.02,
            generator=generator,
        )

    frames = result.frames[0]
    looped = list(frames) + list(reversed(frames[1:-1]))

    out_name = name.replace(".png", ".gif")
    out_path = f"/content/output/{out_name}"
    frame_duration = 1000 // FPS
    frames_to_gif(looped, out_path, target_size=256, duration=frame_duration)
    print(f"✅ {out_name}")

    # Limpar VRAM entre gerações
    torch.cuda.empty_cache()

print(f"\n🎉 Pronto! {len(pngs)} GIFs em /content/output/")

In [ ]:
#@title 8️⃣ Preview de todos os GIFs
import base64
from IPython.display import display, HTML

gifs = sorted([f for f in os.listdir("/content/output") if f.lower().endswith(".gif")])
print(f"👁️ {len(gifs)} GIFs gerados:\n")

html = '<div style="display:flex;flex-wrap:wrap;gap:12px;">'
for name in gifs:
    path = f"/content/output/{name}"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html += f'''
    <div style="text-align:center;background:#1a1a2e;padding:8px;border-radius:8px;">
        <img src="data:image/gif;base64,{b64}" style="width:128px;height:128px;image-rendering:pixelated;">
        <br><small style="color:#ccc;">{name}</small>
    </div>'''
html += '</div>'
display(HTML(html))

In [ ]:
#@title 9️⃣ Baixar tudo como ZIP
import shutil

shutil.make_archive("/content/bilau_animated", "zip", "/content/output")
print("✅ bilau_animated.zip criado!")

try:
    from google.colab import files
    files.download("/content/bilau_animated.zip")
except:
    print("📁 Baixe manualmente: painel esquerdo → bilau_animated.zip")

## 🎛️ Guia de ajuste

### motion_bucket_id (intensidade do movimento)
| Valor | Efeito | Bom pra |
|-------|--------|--------|
| 30-50 | Muito sutil — respiração, piscada | Evoluções calmas (Pequênis, Bilinho) |
| 60-80 | Suave — balanço leve, brilho | Evoluções médias (Bilão, Super-Bilau) |
| 100-130 | Médio — auras, movimento corporal | Evoluções fortes (SSJ, Mega-Bilau) |
| 150-200 | Intenso — efeitos dramáticos | Evoluções épicas (Bileus, Onibilau) |

### noise_aug_strength (fidelidade à imagem original)
| Valor | Efeito |
|-------|--------|
| 0.0 | Máxima fidelidade (quase igual ao original) |
| 0.02 | ✅ **Recomendado** — fiel com movimento natural |
| 0.1+ | Mais liberdade criativa (pode distorcer o sprite) |

### NUM_FRAMES
| Valor | Resultado |
|-------|----------|
| 6-8 | Animação curta, GIF leve |
| 14 | ✅ **Recomendado** — loop suave |
| 20-25 | Animação longa, GIF pesado |

### Dicas
- Use célula 6 pra testar um sprite com diferentes settings antes do batch
- Se o resultado tiver muito movimento, diminua `motion_bucket_id`
- Se perder muito o estilo pixel art, diminua `noise_aug_strength`
- O loop é feito automaticamente (frames + reverse) pra animação cíclica suave

### Depois de baixar
1. Extraia `bilau_animated.zip` em `assets/evolutions/`
2. Renomeie pra 1.gif, 2.gif etc
3. `data-evolutions.js` já aponta pra `assets/evolutions/*.gif`